# Vision Expert — A25 Dataset



In [ ]:
from pathlib import Path
import json, re, time
import base64
import xml.etree.ElementTree as ET

import Levenshtein
from tqdm import tqdm
import asyncio

import csv
from pydantic import BaseModel
from openai import AsyncOpenAI
from typing import List

In [ ]:
DATA_ROOT = Path("A25/input_images")  
OUT_DIR   = Path("outputs/vision_agent_gemma4_run")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DOMAINS = ["Biology", "CompSci", "ICDAR", "MatSci"]  

VISION_MODEL_NAME = "google/gemma-4-31b-it"       # swap to whichever VLM you're testing

client_2 = AsyncOpenAI(
    api_key="vllm",                    
    base_url="http://localhost:8000/v1"
)

SLEEP_BETWEEN_CALLS_SEC = 0 
MAX_RETRIES = 1
RETRY_BACKOFF_SEC = 2.0

print("DATA_ROOT:", DATA_ROOT.resolve())
print("OUT_DIR  :", OUT_DIR.resolve())

CELL_SCHEMA = {
    "type": "object",
    "properties": {
        "cells": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "sr": {"type": "integer"},
                    "er": {"type": "integer"},
                    "sc": {"type": "integer"},
                    "ec": {"type": "integer"},
                    "text": {"type": "string"},
                },
                "required": ["sr", "er", "sc", "ec", "text"],
            },
        }
    },
    "required": ["cells"],
}

class Cell(BaseModel):
    sr: int
    er: int
    sc: int
    ec: int
    text: str


class CellSchema(BaseModel):
    cells: List[Cell]

DATA_ROOT: /Users/isham/project/Table Extraction Researc/kenny_data/input_images
OUT_DIR  : /Users/isham/project/Table Extraction Researc/outputs/vision_agent_gpt_run


In [3]:
prompt = """
    You are a table extractor that accepts a table image and extracts both the cell locations and the cell contents.

    Instruction:
    1. PAY careful attention to the row and column information in the table image.
    2. CORRECTLY extract all mathematical formulas and Greek symbols (e.g., and superscripts/subscripts like x^2 or x_1 directly into normal form).
        - For example, x^2 should be extracted as x2 and x_1 should be extracted as x1.
    3. If a cell span multiple rows or multiple columns, ensure that the sr and er (or sc and ec) are set correctly.
    4. EMPTY cell should be replaced with empty string.
    4. RETURN result in the format below:

    ## SPAN EXAMPLES:
        - Normal cell with no span: sr=0, er=0, sc=0, ec=0
        - Spans 3 columns: sr=0, er=0, sc=1, ec=3
        - Spans 2 rows: sr=1, er=2, sc=0, ec=0
        - Spans 2 cols + 3 rows: sr=0, er=2, sc=1, ec=2

    ## Example output:
        [{"sr": 0, "er": 0, "sc": 0, "ec": 0, "text": "Alloy"},
        {"sr": 0, "er": 0, "sc": 1, "ec": 1, "text": "text1"}, # this was a cell with text: text_1 (subscripts)
        {"sr": 0, "er": 0, "sc": 2, "ec": 2, "text": "text2]"}, # this was a cell with text: text^2 (superscripts)
        {"sr": 0, "er": 0, "sc": 3, "ec": 3, "text": "R2"}]
    
    
    ## OUTPUT FORMAT:
    {"cells": [
    {"sr": 0, "er": 0, "sc": 0, "ec": 0, "text": "cell content"},
    {"sr": 0, "er": 0, "sc": 1, "ec": 1, "text": cell content},
    {"sr": 0, "er": 2, "sc": 1, "ec": 1, "text": cell content}
    .....
    ]}
    
    ONLY RETURN THE OUTPUT. NO OTHER CONTENT SHOULD BE RETURNED

    """

In [ ]:


async def vision_agent(prompt_text: str, encoded_image: str) -> CellSchema:
    response = await client_2.chat.completions.create(
        model=VISION_MODEL_NAME,
        messages=[
            {"role": "system", "content": prompt_text},
            {
                "role": "user",
                "content": [
                    {"type": "text",
                     "text": "Now, please extract the table cells from this image and return in the specified format"},
                    {"type": "image_url",
                     "image_url": {"url": f"data:image/png;base64,{encoded_image}"}},
                ],
            },
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "cell-info",
                "schema": CELL_SCHEMA
            }
        },
        temperature=0.0,
        top_p=0.95,
        max_tokens=16384,
        extra_body={
            "chat_template_kwargs": {"enable_thinking": False}
        }
    )
    return response.choices[0].message.content


def encode_image(image_path: Path) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def normalize_text(text: str) -> str:
    if text is None:
        return ""

    text = str(text).lower()
    text = text.replace("\\times", "x")
    text = text.replace(chr(0x2212), "-").replace(chr(0x2013), "-").replace(chr(0x2014), "-")
    text = re.sub(r"\$\^\{(\d+)\}\$", r"\1", text)
    text = text.replace("$", "")

    return re.sub(r"\s+", " ", text).strip()

def parse_gt_xml(xml_path: Path, normalize: bool = True):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    gt_cells = {}
    for cell in root.findall("cell"):

        sr = int(cell.get("start_row"))
        sc = int(cell.get("start_col"))

        text_node = cell.find("text")
        text = text_node.text if text_node is not None else ""
        text = normalize_text(text) if normalize else (text or "")

        gt_cells[(sr, sc)] = text

    return gt_cells

def parse_pred_json(pred_json):

    pred_cells = {}
    pred_json = pred_json.get("cells", []) if isinstance(pred_json, dict) else pred_json

    for cell in pred_json:
        
        sr = cell.get("sr", cell.get("start-row"))
        sc = cell.get("sc", cell.get("start-col"))

        if sr is None or sc is None:
            continue

        text = normalize_text(cell.get("text", ""))
        pred_cells[(int(sr), int(sc))] = text

    return pred_cells

def compute_accuracy(gt_cells, pred_cells) -> float:
    
    if not gt_cells:
        return 0.0

    total_similarity = 0.0
    count = 0

    for key, gt_text in gt_cells.items():
        pred_text = pred_cells.get(key, "")

        if gt_text == "" and pred_text == "":
            similarity = 1.0
        else:
            dist = Levenshtein.distance(gt_text, pred_text)
            max_len = max(len(gt_text), len(pred_text), 1)
            similarity = 1 - dist / max_len

        total_similarity += similarity
        count += 1

    return total_similarity / max(count, 1)


In [ ]:
def discover_domains(data_root: Path):

    domains = []
    for p in data_root.iterdir():
        if p.is_dir():
            if (p / "images").exists() and (p / "xmls").exists():
                domains.append(p.name)

    return sorted(domains)

if not DOMAINS:
    DOMAINS = discover_domains(DATA_ROOT)

print("DOMAINS:", DOMAINS)

def list_pairs_for_domain(domain_name: str):
    dom_dir = DATA_ROOT / domain_name
    img_dir = dom_dir / "images"
    xml_dir = dom_dir / "xmls"

    img_paths = sorted(img_dir.glob("*.png"))
    pairs = []
    for img_path in img_paths:

        xml_path = xml_dir / f"{img_path.stem}.xml"
        if xml_path.exists():
            pairs.append((img_path, xml_path))
        else:
            pairs.append((img_path, None))
            
    return pairs

DOMAINS: ['Biology', 'CompSci', 'ICDAR', 'MatSci']


In [ ]:

async def process_pair_async(img_path, xml_path, domain_name, preds_dir, errs_dir, semaphore):
    stem = img_path.stem
    pred_file = preds_dir / f"{stem}.json"
    raw_output = None

    if pred_file.exists():
        try:

            pred_json = json.loads(pred_file.read_text(encoding="utf-8"))
            gt_cells = parse_gt_xml(xml_path, normalize=True)
            pred_cells = parse_pred_json(pred_json)

            return {
                "domain": domain_name, 
                "stem": stem, 
                "image_path": str(img_path), 
                "xml_path": str(xml_path),
                "num_gt_cells": len(gt_cells), 
                "num_pred_cells": len(pred_cells),
                "status": "ok",
            }
        except Exception:
            pass  

    if xml_path is None:
        return {
            "domain": domain_name, 
            "stem": stem, 
            "image_path": str(img_path), 
            "xml_path": "",
            "num_gt_cells": 0, 
            "num_pred_cells": 0, 
            "cell_levenshtein_accuracy": 0.0, 
            "status": "missing_xml",
        }

    async with semaphore:
        encoded = encode_image(img_path)
        last_err = None

        for attempt in range(1, MAX_RETRIES + 1):
            try:

                raw_output = await vision_agent(prompt, encoded)
                pred_json = json.loads(raw_output)
                pred_file.write_text(json.dumps(pred_json, ensure_ascii=False, indent=2), encoding="utf-8")
                
                gt_cells = parse_gt_xml(xml_path, normalize=True)
                pred_cells = parse_pred_json(pred_json)
                
                return {
                    "domain": domain_name, 
                    "stem": stem, 
                    "image_path": str(img_path), 
                    "xml_path": str(xml_path),
                    "num_gt_cells": len(gt_cells), 
                    "num_pred_cells": len(pred_cells),
                    "status": "ok",
                }
            except Exception as e:
                last_err = e
                
                if any(err_sig in str(e) for err_sig in ["429", "ResourceExhausted", "RESOURCE_EXHAUSTED"]):
                    
                    wait_time = 45.0 * attempt
                else:
                    wait_time = RETRY_BACKOFF_SEC * attempt
                
                await asyncio.sleep(wait_time)
        else:
           
            err_path = errs_dir / f"{stem}.txt"
            msg = f"ERROR for {stem}\nIMG: {img_path}\nXML: {xml_path}\n\n{repr(last_err)}\n"

            if raw_output:
                msg += "\n\n=== RAW MODEL OUTPUT ===\n" + raw_output

            err_path.write_text(msg, encoding="utf-8")

            return {
                "domain": domain_name, 
                "stem": stem, 
                "image_path": str(img_path), 
                "xml_path": str(xml_path),
                "num_gt_cells": 0, 
                "num_pred_cells": 0, 
                "cell_levenshtein_accuracy": 0.0, 
                "status": "error",
            }
        
# def process_pair_sequential(img_path, xml_path, domain_name, preds_dir, errs_dir):
#     stem = img_path.stem
#     pred_file = preds_dir / f"{stem}.json"
#     raw_output = None


#     if pred_file.exists():
#         try:

#             pred_json = json.loads(pred_file.read_text(encoding="utf-8"))
#             gt_cells = parse_gt_xml(xml_path, normalize=True)
#             pred_cells = parse_pred_json(pred_json)

#             return {
#                 "domain": domain_name, 
#                 "stem": stem, 
#                 "image_path": str(img_path), 
#                 "xml_path": str(xml_path),
#                 "num_gt_cells": len(gt_cells), 
#                 "num_pred_cells": len(pred_cells),
#                 "status": "ok",
#             }
#         except Exception:
#             pass  

#     if xml_path is None:
#         return {
#             "domain": domain_name, 
#             "stem": stem, 
#             "image_path": str(img_path), 
#             "xml_path": "",
#             "num_gt_cells": 0, 
#             "num_pred_cells": 0, 
#             "status": "missing_xml",
#         }

   
#     encoded = encode_image(img_path)
#     last_err = None
    
#     for attempt in range(1, MAX_RETRIES + 1):
#         try:

#             raw_output = vision_agent(prompt, encoded)
#             pred_json = raw_output.model_dump()
            
            
#             pred_file.write_text(json.dumps(pred_json, ensure_ascii=False, indent=2), encoding="utf-8")
            
#             gt_cells = parse_gt_xml(xml_path, normalize=True)
#             pred_cells = parse_pred_json(pred_json)
            
#             return {
#                 "domain": domain_name, 
#                 "stem": stem, 
#                 "image_path": str(img_path), 
#                 "xml_path": str(xml_path),
#                 "num_gt_cells": len(gt_cells), 
#                 "num_pred_cells": len(pred_cells),
#                 "status": "ok",
#             }
            
#         except Exception as e:
#             last_err = e
            
#             print(f"Error for {stem}: {repr(last_err)}")
#             time.sleep(2.0 * attempt)
            

#     err_path = errs_dir / f"{stem}.txt"
#     msg = f"ERROR for {stem}\nIMG: {img_path}\nXML: {xml_path}\n\n{repr(last_err)}\n"

#     if raw_output:
#         if hasattr(raw_output, "model_dump_json"):
#             msg += "\n\n=== RAW MODEL OUTPUT ===\n"
#             msg += raw_output.model_dump_json(indent=2)
#         else:
#             msg += "\n\n=== RAW MODEL OUTPUT ===\n" + str(raw_output)

#     err_path.write_text(msg, encoding="utf-8")

#     return {
#         "domain": domain_name, 
#         "stem": stem, 
#         "image_path": str(img_path), 
#         "xml_path": str(xml_path),
#         "num_gt_cells": 0, 
#         "num_pred_cells": 0,
#         "status": "error",
#     }

In [ ]:

async def run_domain_async(domain_name: str, semaphore: asyncio.Semaphore):
    pairs = list_pairs_for_domain(domain_name)
    dom_out = OUT_DIR / domain_name
    dom_out.mkdir(parents=True, exist_ok=True)

    preds_dir = dom_out / "predictions"
    errs_dir  = dom_out / "errors"
    preds_dir.mkdir(exist_ok=True)
    errs_dir.mkdir(exist_ok=True)

    results_csv = dom_out / "results.csv"

    done = set()
    if results_csv.exists():
        with open(results_csv, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                done.add(row["stem"])

    write_header = not results_csv.exists()
    
    
    pairs_to_process = [p for p in pairs if p[0].stem not in done]

    if not pairs_to_process:
        print(f"All tables in {domain_name} already processed.")
        return results_csv

    with open(results_csv, "a", newline="", encoding="utf-8") as f:
        fieldnames = [
            "domain", 
            "stem", 
            "image_path", 
            "xml_path",
            "num_gt_cells", 
            "num_pred_cells",
            "status",
        ]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()

        
        tasks = [
            process_pair_async(img_path, xml_path, domain_name, preds_dir, errs_dir, semaphore)
            for img_path, xml_path in pairs_to_process
        ]

        
        for future in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc=f"{domain_name} (Concurrent)"):
            row_data = await future
            writer.writerow(row_data)
            f.flush()

    return results_csv


# def run_domain_sequential(domain_name: str):

#     pairs = list_pairs_for_domain(domain_name)
#     dom_out = OUT_DIR / domain_name
#     dom_out.mkdir(parents=True, exist_ok=True)

#     preds_dir = dom_out / "predictions"
#     errs_dir  = dom_out / "errors"
#     preds_dir.mkdir(exist_ok=True)
#     errs_dir.mkdir(exist_ok=True)

#     results_csv = dom_out / "results.csv"

#     done = set()
#     if results_csv.exists():
#         with open(results_csv, "r", newline="", encoding="utf-8") as f:
#             reader = csv.DictReader(f)
#             for row in reader:
#                 done.add(row["stem"])

#     write_header = not results_csv.exists()
#     pairs_to_process = [p for p in pairs if p[0].stem not in done]

#     if not pairs_to_process:
#         print(f"All tables in {domain_name} already processed.")
#         return results_csv

#     with open(results_csv, "a", newline="", encoding="utf-8") as f:
#         fieldnames = [
#             "domain", 
#             "stem", 
#             "image_path", 
#             "xml_path",
#             "num_gt_cells", 
#             "num_pred_cells",
#             "status",
#         ]
#         writer = csv.DictWriter(f, fieldnames=fieldnames)
#         if write_header:
#             writer.writeheader()

#         for img_path, xml_path in tqdm(pairs_to_process, desc=f"{domain_name} (Sequential)"):
#             row_data = process_pair_sequential(img_path, xml_path, domain_name, preds_dir, errs_dir)
#             writer.writerow(row_data)
#             f.flush()  

#     return results_csv

In [ ]:
MAX_CONCURRENT_TASKS = 5
shared_semaphore = asyncio.Semaphore(MAX_CONCURRENT_TASKS)

domain_results = []
for d in DOMAINS:
    csv_path = await run_domain_async(d, shared_semaphore)
    domain_results.append(csv_path)

domain_results

# domain_results = []
# for d in DOMAINS:
#     csv_path = run_domain_sequential(d)
#     domain_results.append(csv_path)

# domain_results

MatSci (Sequential): 100%|██████████| 49/49 [06:41<00:00,  8.19s/it]


[PosixPath('outputs/vision_agent_gpt_run/Biology/results.csv'),
 PosixPath('outputs/vision_agent_gpt_run/CompSci/results.csv'),
 PosixPath('outputs/vision_agent_gpt_run/ICDAR/results.csv'),
 PosixPath('outputs/vision_agent_gpt_run/MatSci/results.csv')]